# Notebook 05 — Final Validation & Tableau Prep

This notebook serves as the final quality assurance gate. It validates the integrity of the three processed CSV files and prepares them for seamless connection to Tableau. Specifically, it enforces strict data types, converts booleans to integers for easier calculation in Tableau, and programmatically generates a data dictionary.

## Section 1: Reload and Validate `master_fact.csv`

In [1]:
import os
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────────
PROCESSED_PATH = '../data/processed/'
DOCS_PATH      = '../docs/'
os.makedirs(DOCS_PATH, exist_ok=True)

print("Validation environment ready.")

Validation environment ready.


In [2]:
fact_path = os.path.join(PROCESSED_PATH, 'master_fact.csv')
master_fact = pd.read_csv(fact_path)

print(f"Loaded master_fact: {master_fact.shape[0]:,} rows, {master_fact.shape[1]} columns.")

# ── Assertions ─────────────────────────────────────────────────────────────────
try:
    # 1. Row count
    assert len(master_fact) == 27304, f"FAIL: Expected 27,304 rows, got {len(master_fact):,}"
    
    # 2. positionOrder nulls
    assert master_fact['positionOrder'].isna().sum() == 0, "FAIL: positionOrder contains null values"
    
    # 3. \N strings
    # Convert to string and search for exact \N pattern
    has_slash_n = master_fact.apply(lambda x: x.astype(str).str.contains(r'^\\N$').any()).any()
    assert not has_slash_n, "FAIL: Literal \\N strings detected in dataset"
    
    # 4. Year range
    year_min, year_max = master_fact['year'].min(), master_fact['year'].max()
    assert 1950 <= year_min <= 2026, f"FAIL: year_min {year_min} outside range [1950, 2026]"
    assert 1950 <= year_max <= 2026, f"FAIL: year_max {year_max} outside range [1950, 2026]"
    
    # 5. grid_to_finish_delta
    assert 'grid_to_finish_delta' in master_fact.columns, "FAIL: grid_to_finish_delta column missing"
    assert master_fact['grid_to_finish_delta'].notna().any(), "FAIL: grid_to_finish_delta is empty"
    
    # 6. Booleans (check if they are bool or int or string representation of bool)
    bool_cols = ['is_finisher', 'is_dnf', 'is_podium', 'is_win', 'is_pole']
    for col in bool_cols:
        assert col in master_fact.columns, f"FAIL: Missing boolean column {col}"
    
    # 7. Key name nulls
    assert master_fact['constructor_name'].isna().sum() == 0, "FAIL: constructor_name contains nulls"
    assert master_fact['circuit_name'].isna().sum() == 0, "FAIL: circuit_name contains nulls"
    
    # 8. DNF category
    assert 'dnf_category' in master_fact.columns, "FAIL: dnf_category missing"
    expected_dnfs = {'Finished', 'Accident', 'Mechanical', 'DNQ', 'Other'}
    found_dnfs = set(master_fact['dnf_category'].unique())
    assert found_dnfs.issubset(expected_dnfs), f"FAIL: Unexpected DNF categories: {found_dnfs - expected_dnfs}"

    print("SECTION 1: master_fact.csv validation PASS")
except AssertionError as e:
    print(str(e))
    raise

Loaded master_fact: 27,304 rows, 62 columns.


SECTION 1: master_fact.csv validation PASS


## Section 2: Validate `constructor_season_kpis.csv`

In [3]:
kpi_path = os.path.join(PROCESSED_PATH, 'constructor_season_kpis.csv')
kpis = pd.read_csv(kpi_path)

try:
    # 1. Uniqueness
    duplicates = kpis.duplicated(subset=['constructorId', 'year']).sum()
    assert duplicates == 0, f"FAIL: Found {duplicates} duplicate constructor-season entries"
    
    # 2. Points efficiency
    assert pd.api.types.is_numeric_dtype(kpis['points_efficiency']), "FAIL: points_efficiency is not numeric"
    
    # 3. DNF rate
    assert kpis['dnf_rate'].between(0, 1).all(), "FAIL: dnf_rate outside [0, 1] range"
    
    # 4. Volatility
    assert (kpis['points_volatility'].dropna() >= 0).all(), "FAIL: Negative points_volatility detected"

    print("SECTION 2: constructor_season_kpis.csv validation PASS")
    print(f"Summary: {kpis['year'].min()}-{kpis['year'].max()}, "
          f"{kpis['constructorId'].nunique()} constructors, {len(kpis)} seasons.")
except AssertionError as e:
    print(str(e))
    raise

SECTION 2: constructor_season_kpis.csv validation PASS
Summary: 1950-2026, 213 constructors, 1132 seasons.


## Section 3: Validate `circuit_strategy_profile.csv`

In [4]:
profile_path = os.path.join(PROCESSED_PATH, 'circuit_strategy_profile.csv')
profile = pd.read_csv(profile_path)

try:
    # 1. Row count
    assert len(profile) == 78, f"FAIL: Expected 78 circuits, got {len(profile)}"
    
    # 2. Lat/Lng range
    # Note: circuits.csv in data/raw/ has lat/lng. If joined correctly, they should be valid.
    # Some circuits might not have these if the join failed, check for nulls if they are expected.
    # For now, validate range if they exist.
    if 'lat' in profile.columns and 'lng' in profile.columns:
        assert profile['lat'].between(-90, 90).all(), "FAIL: latitude out of range"
        assert profile['lng'].between(-180, 180).all(), "FAIL: longitude out of range"
    
    # 3. Cluster Label (Note: requires Notebook 04 to have run)
    assert 'cluster_label' in profile.columns, "FAIL: cluster_label missing (Run Notebook 04 first)"
    n_clusters = profile['cluster_label'].nunique()
    assert n_clusters == 3, f"FAIL: Expected 3 distinct clusters, found {n_clusters}"
    
    # 4. Strategy positions
    assert 'avg_1stop_position' in profile.columns, "FAIL: avg_1stop_position missing"
    assert 'avg_2stop_position' in profile.columns, "FAIL: avg_2stop_position missing"

    print("SECTION 3: circuit_strategy_profile.csv validation PASS")
    print("Cluster Distribution:")
    print(profile['cluster_label'].value_counts())
except AssertionError as e:
    print(str(e))
    raise

SECTION 3: circuit_strategy_profile.csv validation PASS
Cluster Distribution:
cluster_label
Mixed                  19
Strategy-Dominant      10
Qualifying-Dominant     5
Name: count, dtype: int64


## Section 4: Tableau-Specific Formatting

In [5]:
print("Applying Tableau-specific formatting...")

# ── Master Fact Formatting ─────────────────────────────────────────────────────
master_fact['year']           = master_fact['year'].astype('int64')
master_fact['grid']           = master_fact['grid'].fillna(0).astype('int64')
master_fact['positionOrder']  = master_fact['positionOrder'].astype('int64')
master_fact['points']         = master_fact['points'].astype('float64')

# Convert booleans to 0/1 integers for Tableau calculations
bool_cols = ['is_finisher', 'is_dnf', 'is_podium', 'is_win', 'is_pole', 'is_pitlane_start']
for col in bool_cols:
    if col in master_fact.columns:
        master_fact[col] = master_fact[col].fillna(0).astype(int)

# Driver name display: Surname, Forename
master_fact['driver_name_display'] = master_fact['surname'].str.strip() + ", " + master_fact['forename'].str.strip()

# Constructor short names mapping (for tight layouts)
constructor_map = {
    'Red Bull': 'RBR', 'Mercedes': 'MER', 'Ferrari': 'FER', 
    'McLaren': 'MCL', 'Alpine F1 Team': 'ALP', 'Aston Martin': 'AST',
    'AlphaTauri': 'ALT', 'Williams': 'WIL', 'Haas F1 Team': 'HAS',
    'Alfa Romeo': 'ALF', 'Racing Point': 'RP', 'Renault': 'REN',
    'Toro Rosso': 'STR', 'Force India': 'VJM', 'Sauber': 'SAU'
}
master_fact['constructor_short'] = master_fact['constructor_name'].map(constructor_map).fillna(master_fact['constructor_name'])

# Era classification for filtering
def classify_era(year):
    if year < 2006:
        return 'V10 Era'
    elif year < 2014:
        return 'V8 Era'
    else:
        return 'Turbo Hybrid Era'

master_fact['era'] = master_fact['year'].apply(classify_era)

# Grid Delta Category for Dashboard 3
def categorize_delta(delta):
    if pd.isna(delta):
        return 'Unknown'
    elif delta > 2:
        return 'Gained 3+'
    elif delta > 0:
        return 'Gained 1-2'
    elif delta == 0:
        return 'Held Position'
    else:
        return 'Lost Positions'

master_fact['grid_delta_category'] = master_fact['grid_to_finish_delta'].apply(categorize_delta)

# Stop count bucket for Dashboard 2
def bucket_stops(stops):
    if pd.isna(stops):
        return 'Unknown'
    elif stops >= 3:
        return '3+ stops'
    elif stops == 2:
        return '2 stops'
    else:
        return '1 stop'

if 'stop_count' in master_fact.columns:
    master_fact['stop_count_bucket'] = master_fact['stop_count'].apply(bucket_stops)

# Convert pit times to seconds for readability
if 'avg_pit_ms' in master_fact.columns:
    master_fact['avg_pit_seconds'] = master_fact['avg_pit_ms'] / 1000

if 'fastest_pit_ms' in master_fact.columns:
    master_fact['fastest_pit_seconds'] = master_fact['fastest_pit_ms'] / 1000

# Strip whitespace from key text fields
master_fact['circuit_name'] = master_fact['circuit_name'].str.strip()
master_fact['constructor_name'] = master_fact['constructor_name'].str.strip()

# ── KPI Formatting ────────────────────────────────────────────────────────────
rate_cols = ['points_efficiency', 'podium_rate', 'win_rate', 'pole_to_win_rate', 'dnf_rate']
for col in rate_cols:
    if col in kpis.columns:
        kpis[col] = kpis[col].round(4)

kpis['constructor_name'] = kpis['constructor_name'].str.strip()

# Add era to KPIs
kpis['era'] = kpis['year'].apply(classify_era)

# ── Circuit Profile Enhancements ──────────────────────────────────────────────
# Add qualifying_lock_in_score (higher = qualifying matters more)
# Based on low avg_delta and high qualifying_gap
if 'avg_delta' in profile.columns and 'avg_qualifying_gap' in profile.columns:
    # Normalize to 0-100 scale
    delta_norm = (profile['avg_delta'].max() - profile['avg_delta']) / (profile['avg_delta'].max() - profile['avg_delta'].min())
    gap_norm = profile['avg_qualifying_gap'] / profile['avg_qualifying_gap'].max()
    profile['qualifying_lock_in_score'] = ((delta_norm * 0.6 + gap_norm * 0.4) * 100).round(1)

# Add optimal_stop_count (most common successful strategy)
if 'best_strategy_stops' in profile.columns:
    profile['optimal_stop_count'] = profile['best_strategy_stops'].fillna(2).astype(int)
elif 'avg_1stop_position' in profile.columns and 'avg_2stop_position' in profile.columns:
    # Choose strategy with better average position
    profile['optimal_stop_count'] = profile.apply(
        lambda row: 1 if pd.notna(row['avg_1stop_position']) and 
                        (pd.isna(row['avg_2stop_position']) or row['avg_1stop_position'] < row['avg_2stop_position'])
                    else 2,
        axis=1
    )

# Add compound_bias placeholder (for future tyre data integration)
if 'compound_bias' not in profile.columns:
    profile['compound_bias'] = 0  # Neutral baseline

print("Formatting complete.")
print(f"\nTableau-ready columns added:")
print(f"  - era (V10/V8/Turbo Hybrid)")
print(f"  - grid_delta_category (Gained 3+, Gained 1-2, Held, Lost)")
print(f"  - stop_count_bucket (1/2/3+ stops)")
print(f"  - avg_pit_seconds (converted from ms)")
print(f"  - qualifying_lock_in_score (0-100 scale)")
print(f"  - optimal_stop_count (1 or 2)")

Applying Tableau-specific formatting...
Formatting complete.

Tableau-ready columns added:
  - era (V10/V8/Turbo Hybrid)
  - grid_delta_category (Gained 3+, Gained 1-2, Held, Lost)
  - stop_count_bucket (1/2/3+ stops)
  - avg_pit_seconds (converted from ms)
  - qualifying_lock_in_score (0-100 scale)
  - optimal_stop_count (1 or 2)


## Section 5: Generate Data Dictionary

In [6]:
print("Generating data dictionary...")

descriptions = {
    'resultId': 'Unique internal identifier for each race entry',
    'raceId': 'Foreign key identifying the specific race event',
    'driverId': 'Unique identifier for the driver',
    'constructorId': 'Unique identifier for the car manufacturer',
    'grid': 'Starting position on the race grid (1=Pole, 0=Pit Lane)',
    'positionOrder': 'Final finishing classification order (1-max)',
    'points': 'Championship points awarded for the race',
    'is_finisher': 'Binary flag: 1 if driver completed race distance, 0 otherwise',
    'is_dnf': 'Binary flag: 1 if driver did not finish, 0 otherwise',
    'is_win': 'Binary flag: 1 if driver finished 1st, 0 otherwise',
    'is_pole': 'Binary flag: 1 if driver started from pole position, 0 otherwise',
    'year': 'Calendar year of the race season',
    'circuit_name': 'Official name of the race track',
    'constructor_name': 'Commercial name of the F1 team',
    'dnf_category': 'Classification of DNF reason (Mechanical, Accident, etc.)',
    'grid_to_finish_delta': 'Net positions gained (Grid - PositionOrder)',
    'avg_pit_ms': 'Average time spent in pit stops (milliseconds)',
    'qualifying_gap_ms': 'Time gap between driver\'s best qualifying lap and pole time',
    'overtaking_score': 'Circuit difficulty tier based on lap time variance',
    'cluster_label': 'Strategy archetype assigned to the circuit (e.g., Strategy-Dominant)',
    'driver_name_display': 'Formatted driver name for labels (Surname, Forename)',
    'constructor_short': 'Abbreviated constructor name for tight visualization layouts'
}

dict_lines = ["# Data Dictionary — F1 Race Strategy Intelligence\n"]
dict_lines.append("## master_fact.csv\n")
dict_lines.append("| Column Name | Dtype | Description | Example Value | Null Count |")
dict_lines.append("| :--- | :--- | :--- | :--- | :--- |")

for col in master_fact.columns:
    dtype = str(master_fact[col].dtype)
    desc  = descriptions.get(col, "TBD")
    example = str(master_fact[col].iloc[0])
    nulls = master_fact[col].isna().sum()
    dict_lines.append(f"| {col} | {dtype} | {desc} | {example} | {nulls} |")

dict_path = os.path.join(DOCS_PATH, 'data_dictionary.md')
with open(dict_path, 'w') as f:
    f.write("\n".join(dict_lines))

print(f"Data dictionary saved to {dict_path}")

Generating data dictionary...
Data dictionary saved to ../docs/data_dictionary.md


## Section 6: Final Export

In [7]:
print("Exporting Tableau-ready files...")

export_info = []

def export_and_log(df, filename):
    path = os.path.join(PROCESSED_PATH, filename)
    df.to_csv(path, index=False)
    size_kb = os.path.getsize(path) / 1024
    export_info.append({
        'File Name': filename,
        'Rows': len(df),
        'Cols': len(df.columns),
        'Size (KB)': f"{size_kb:.2f}"
    })

export_and_log(master_fact, 'master_fact.csv')
export_and_log(kpis, 'constructor_season_kpis.csv')
export_and_log(profile, 'circuit_strategy_profile.csv')

display(pd.DataFrame(export_info))
print("\nHANDOFF READY — connect Tableau to data/processed/")
print("All checks passed. Tableau connection ready.")

Exporting Tableau-ready files...


,File Name,Rows,Cols,Size (KB)
0,master_fact.csv,27304,62,9063.77
1,constructor_season_kpis.csv,1132,17,124.60
2,circuit_strategy_profile.csv,78,19,8.93



HANDOFF READY — connect Tableau to data/processed/
All checks passed. Tableau connection ready.


## Section 7: Tableau Setup Instructions

### Data Connections
1. **Master Analysis**: Connect to `master_fact.csv`. Use this for entry-level analysis (Grid vs Finish, DNF trends).
2. **Team Performance**: Connect to `constructor_season_kpis.csv`. Use for year-over-year efficiency tracking.
3. **Circuit Archetypes**: Connect to `circuit_strategy_profile.csv`. Use for geographic mapping and cluster-based filtering.

### Dimension & Measure Setup
- **Dimensions**: `year`, `raceId`, `driverId`, `constructorId`, `circuitId`, `dnf_category`, `cluster_label`, `constructor_short`.
- **Measures**: `grid`, `positionOrder`, `points`, `is_win` (SUM), `is_pole` (SUM), `grid_to_finish_delta`, `avg_pit_ms`.

### Calculated Fields (Recommended)
- **Pole Win %**: `SUM([is_win]) / SUM([is_pole])` — Filter where `is_pole = 1`.
- **DNF %**: `SUM([is_dnf]) / COUNT([resultId])`
- **Avg Pit Duration (sec)**: `AVG([avg_pit_ms]) / 1000`
- **Points Efficiency Rank**: `RANK(SUM([points_efficiency]), 'desc')`

### Recommended Filters
- **Hybrid Era Toggle**: Filter `year >= 2014` to focus on modern power unit reliability.
- **Mid-field Constructors**: Group constructors by their championship ranking (4th-7th) to isolate the decision-maker's context.
- **Circuit Cluster**: Filter by `cluster_label` to compare Strategy-Dominant vs. Qualifying-Dominant outcomes.